# E-Commerce User Behavior Analytics and Personalized Product Recommendation using Big Data
**Big Data Mini Project (extension of Experiment 1 case study)**

Team members: *Student 1, Student 2, Student 3*

**Pipeline:** Raw CSV -> PySpark ETL -> Parquet data lake -> Spark SQL analytics -> K-Means segmentation -> ALS recommender -> Scalability benchmarks

**Dataset:** Kaggle "eCommerce behavior data from multi-category store" (columns: `event_time, event_type, product_id, category_id, category_code, brand, price, user_id, user_session`).
If the file is not found, the notebook generates a synthetic dataset with the same schema so every cell still runs.

## 0. Setup

In [ ]:
# Run in Colab / local. Uncomment the next line if pyspark is not installed.
# !pip install -q pyspark matplotlib pandas

import os, time, random
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder
         .appName("EcomBehaviorBigData")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

RAW_PATH = "/content/2019-Oct.csv"      # change to your downloaded Kaggle file
SAMPLE_FRACTION = 0.1               # use 0.1 on laptops/Colab; 1.0 for full data
LAKE = "lake"                       # data lake folder
os.makedirs(LAKE, exist_ok=True)

## 1. Data ingestion (Volume, Variety)
Read raw CSV with an explicit schema (faster than `inferSchema` on big files). If the Kaggle file is missing, a synthetic dataset is created.

In [ ]:
from pyspark.sql.types import *

schema = StructType([
    StructField("event_time", StringType()),
    StructField("event_type", StringType()),
    StructField("product_id", LongType()),
    StructField("category_id", LongType()),
    StructField("category_code", StringType()),
    StructField("brand", StringType()),
    StructField("price", DoubleType()),
    StructField("user_id", LongType()),
    StructField("user_session", StringType()),
])

def make_synthetic(n_rows=300_000, n_users=5_000, n_products=2_000, seed=42):
    random.seed(seed)
    cats = ["electronics.smartphone", "electronics.audio", "appliances.kitchen", "computers.notebook",
            "apparel.shoes", "furniture.living_room", "kids.toys", "sport.bicycle"]
    brands = ["samsung", "apple", "xiaomi", "lg", "sony", "bosch", "nike", "lenovo", None]
    prod = {p: (random.choice(cats), random.choice(brands), round(random.uniform(5, 900), 2)) for p in range(n_products)}
    rows = []
    for _ in range(n_rows):
        u = int(random.paretovariate(1.2)) % n_users + 1000
        p = int(random.paretovariate(1.1)) % n_products
        c, b, pr = prod[p]
        r = random.random()
        et = "view" if r < 0.90 else ("cart" if r < 0.97 else "purchase")
        ts = f"2019-10-{random.randint(1,30):02d} {random.choice([8,9,10,11,12,13,14,15,16,17,18,19,20,21,22])}:{random.randint(0,59):02d}:00 UTC"
        rows.append((ts, et, p, p, c, b, pr, u, f"s{u}_{random.randint(1,5)}"))
    return spark.createDataFrame(rows, schema)

if os.path.exists(RAW_PATH):
    raw = spark.read.csv(RAW_PATH, header=True, schema=schema)
    if SAMPLE_FRACTION < 1.0:
        raw = raw.sample(False, SAMPLE_FRACTION, seed=42)
    print("Loaded real dataset")
else:
    raw = make_synthetic()
    print("Real file not found -> using synthetic dataset")

print("Rows:", raw.count())
raw.printSchema()
raw.show(5, truncate=False)

## 2. Cleaning and ETL (Veracity)
Steps: drop duplicates, remove null users/products, parse timestamps, remove invalid prices, fill missing brand/category, derive `date`, `hour`, `main_category`.

In [ ]:
clean = (raw.dropDuplicates()
    .filter(F.col("user_id").isNotNull() & F.col("product_id").isNotNull())
    .filter(F.col("price") > 0)
    .withColumn("event_time", F.to_timestamp(F.regexp_replace("event_time", " UTC", "")))
    .withColumn("date", F.to_date("event_time"))
    .withColumn("hour", F.hour("event_time"))
    .fillna({"brand": "unknown", "category_code": "unknown.unknown"})
    .withColumn("main_category", F.split("category_code", r"\.").getItem(0)))

print("Before:", raw.count(), " After:", clean.count())

## 3. Distributed storage: CSV vs partitioned Parquet
Parquet is columnar and compressed, which means smaller files and faster analytical queries. Data is partitioned by `date`.

In [ ]:
clean.write.mode("overwrite").partitionBy("date").parquet(f"{LAKE}/clean_events")
df = spark.read.parquet(f"{LAKE}/clean_events").cache()
df.createOrReplaceTempView("events")
print("Parquet rows:", df.count())

def folder_size_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return round(total / 1024 / 1024, 2)

# Save the same data as CSV for comparison
clean.drop("date").write.mode("overwrite").option("header", True).csv(f"{LAKE}/clean_events_csv")

size_csv, size_pq = folder_size_mb(f"{LAKE}/clean_events_csv"), folder_size_mb(f"{LAKE}/clean_events")
print(f"CSV size: {size_csv} MB | Parquet size: {size_pq} MB")

def timed(fn):
    t = time.time(); fn(); return round(time.time() - t, 2)

t_csv = timed(lambda: spark.read.option("header", True).csv(f"{LAKE}/clean_events_csv").groupBy("brand").count().collect())
t_pq  = timed(lambda: spark.read.parquet(f"{LAKE}/clean_events").groupBy("brand").count().collect())
print(f"Group-by time -> CSV: {t_csv}s | Parquet: {t_pq}s")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].bar(["CSV", "Parquet"], [size_csv, size_pq], color=["#d9534f", "#5cb85c"]); ax[0].set_title("Storage size (MB)")
ax[1].bar(["CSV", "Parquet"], [t_csv, t_pq], color=["#d9534f", "#5cb85c"]); ax[1].set_title("Query time (s)")
plt.tight_layout(); plt.show()

## 4. User behavior analytics (Spark SQL)
### 4.1 Event distribution and conversion funnel

In [ ]:
funnel = spark.sql('''
SELECT event_type, COUNT(*) AS events, COUNT(DISTINCT user_session) AS sessions, COUNT(DISTINCT user_id) AS users
FROM events GROUP BY event_type ORDER BY events DESC
''').toPandas()
print(funnel)

s = dict(zip(funnel.event_type, funnel.sessions))
v, c, p = s.get("view", 0), s.get("cart", 0), s.get("purchase", 0)
print(f"View->Cart: {c/v*100:.2f}% | Cart->Purchase: {p/c*100 if c else 0:.2f}% | View->Purchase: {p/v*100:.2f}%")

plt.figure(figsize=(6, 3.5))
plt.barh(["Purchase", "Cart", "View"], [p, c, v], color=["#5cb85c", "#f0ad4e", "#5bc0de"])
plt.title("Conversion funnel (unique sessions)"); plt.tight_layout(); plt.show()

### 4.2 Cart abandonment

In [ ]:
ab = spark.sql('''
SELECT
  COUNT(DISTINCT CASE WHEN event_type='cart' THEN user_session END) AS carted,
  COUNT(DISTINCT CASE WHEN event_type='purchase' THEN user_session END) AS purchased
FROM events
''').toPandas()
carted, bought = int(ab.carted[0]), int(ab.purchased[0])
print(f"Cart abandonment rate (approx.): {(1 - bought / carted) * 100:.2f}%")

### 4.3 Peak hours and daily trend

In [ ]:
hourly = spark.sql("SELECT hour, event_type, COUNT(*) AS n FROM events GROUP BY hour, event_type").toPandas()
pivot = hourly.pivot(index="hour", columns="event_type", values="n").fillna(0)
pivot.plot(kind="line", figsize=(9, 4), marker="o", title="Events by hour of day")
plt.ylabel("events"); plt.show()

daily = spark.sql("SELECT date, COUNT(*) AS events FROM events GROUP BY date ORDER BY date").toPandas()
daily.plot(x="date", y="events", figsize=(9, 3.5), title="Daily event volume", legend=False); plt.show()

### 4.4 Top categories, brands and revenue

In [ ]:
top_cat = spark.sql('''
SELECT main_category, COUNT(*) AS views FROM events WHERE event_type='view'
GROUP BY main_category ORDER BY views DESC LIMIT 10''').toPandas()

top_brand = spark.sql('''
SELECT brand, ROUND(SUM(price),2) AS revenue, COUNT(*) AS orders
FROM events WHERE event_type='purchase' AND brand <> 'unknown'
GROUP BY brand ORDER BY revenue DESC LIMIT 10''').toPandas()

print(top_cat); print(top_brand)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
top_cat.plot.barh(x="main_category", y="views", ax=ax[0], legend=False, title="Top categories by views"); ax[0].invert_yaxis()
top_brand.plot.barh(x="brand", y="revenue", ax=ax[1], legend=False, color="#5cb85c", title="Top brands by revenue"); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()

### 4.5 Sessionization with window functions
Average session length and events per session, computed in parallel with window functions.

In [ ]:
sess = (df.groupBy("user_session")
          .agg(F.min("event_time").alias("start"), F.max("event_time").alias("end"),
               F.count("*").alias("events"),
               F.max(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("converted"))
          .withColumn("duration_min", (F.col("end").cast("long") - F.col("start").cast("long")) / 60))

sess.agg(F.avg("duration_min").alias("avg_duration_min"),
         F.avg("events").alias("avg_events_per_session"),
         F.avg("converted").alias("session_conversion_rate")).show()

# Previous-event / next-event analysis with a window
w = Window.partitionBy("user_session").orderBy("event_time")
transitions = (df.withColumn("next_event", F.lead("event_type").over(w))
                 .groupBy("event_type", "next_event").count().orderBy(F.desc("count")))
transitions.show(10)

## 5. Customer segmentation: RFM + K-Means (MLlib)
Recency, Frequency and Monetary value are computed for all purchasing users, scaled, then clustered.

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

max_date = df.agg(F.max("date")).collect()[0][0]

rfm = (df.filter("event_type = 'purchase'")
         .groupBy("user_id")
         .agg(F.datediff(F.lit(max_date), F.max("date")).alias("recency"),
              F.countDistinct("user_session").alias("frequency"),
              F.round(F.sum("price"), 2).alias("monetary")))
print("Purchasing users:", rfm.count())

vec = VectorAssembler(inputCols=["recency", "frequency", "monetary"], outputCol="raw_features").transform(rfm)
scaled = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True).fit(vec).transform(vec)

# Choose k using the silhouette score
best_k, best_s = 2, -1
for k in range(2, 7):
    m = KMeans(k=k, seed=42).fit(scaled)
    s = ClusteringEvaluator().evaluate(m.transform(scaled))
    print(f"k={k}  silhouette={s:.3f}")
    if s > best_s: best_k, best_s = k, s
print("Best k:", best_k)

km = KMeans(k=best_k, seed=42).fit(scaled)
seg = km.transform(scaled)
profile = (seg.groupBy("prediction")
              .agg(F.count("*").alias("users"), F.round(F.avg("recency"), 1).alias("avg_recency"),
                   F.round(F.avg("frequency"), 2).alias("avg_frequency"), F.round(F.avg("monetary"), 2).alias("avg_monetary"))
              .orderBy("prediction"))
profile.show()

In [ ]:
pdf = profile.toPandas()
pdf.set_index("prediction")[["avg_recency", "avg_frequency", "avg_monetary"]].plot(
    kind="bar", subplots=True, figsize=(9, 6), legend=False, title=["Avg recency (days)", "Avg frequency", "Avg monetary"])
plt.tight_layout(); plt.show()

# Interpretation guide (edit labels after looking at your cluster table):
#  - low recency, high frequency, high monetary -> Champions
#  - medium recency, medium frequency          -> Loyal / Potential loyalists
#  - high recency (long ago), low frequency    -> At-risk / Lost
# Business action: reward Champions, send discounts to At-risk users, win-back emails to Lost users.

## 6. Recommendation engine: ALS collaborative filtering
Implicit feedback weights: **view = 1, cart = 3, purchase = 5**. Spark's ALS factorises the huge sparse user-item matrix in a distributed way.

In [ ]:
from pyspark.ml.recommendation import ALS

ratings = (df.withColumn("w", F.when(F.col("event_type") == "purchase", 5.0)
                               .when(F.col("event_type") == "cart", 3.0).otherwise(1.0))
             .groupBy("user_id", "product_id").agg(F.sum("w").alias("rating")))

# Keep users/products with some activity (reduces noise and sparsity)
active_users = ratings.groupBy("user_id").count().filter("count >= 3").select("user_id")
ratings = ratings.join(active_users, "user_id")
ratings = ratings.withColumn("user_id", F.col("user_id").cast("int")).withColumn("product_id", F.col("product_id").cast("int"))
ratings.cache()
print("User-item pairs:", ratings.count())

train, test = ratings.randomSplit([0.8, 0.2], seed=42)

als = ALS(userCol="user_id", itemCol="product_id", ratingCol="rating",
          implicitPrefs=True, rank=20, regParam=0.1, alpha=10.0, maxIter=10,
          coldStartStrategy="drop", nonnegative=True)
t = time.time(); als_model = als.fit(train); print("ALS trained in", round(time.time() - t, 1), "s")

recs = als_model.recommendForAllUsers(5)
recs.show(5, truncate=False)

### 6.1 Evaluation: Precision@K and Recall@K
A recommendation is a "hit" if the user later interacted strongly (cart or purchase, rating >= 3) with that product in the held-out test set.

In [ ]:
K = 10
topk = (als_model.recommendForAllUsers(K)
        .select("user_id", F.explode("recommendations").alias("rec"))
        .select("user_id", F.col("rec.product_id").alias("product_id")))

relevant = test.filter("rating >= 3").select("user_id", "product_id").distinct()
n_rel = relevant.groupBy("user_id").agg(F.count("*").alias("n_rel"))

hits = (topk.join(relevant, ["user_id", "product_id"]).groupBy("user_id").agg(F.count("*").alias("hits")))
eval_df = n_rel.join(hits, "user_id", "left").fillna(0, subset=["hits"])

m = eval_df.agg(F.avg(F.col("hits") / K).alias("precision_at_k"),
                F.avg(F.col("hits") / F.col("n_rel")).alias("recall_at_k")).collect()[0]
print(f"Precision@{K}: {m['precision_at_k']:.4f} | Recall@{K}: {m['recall_at_k']:.4f} | evaluated users: {eval_df.count()}")

# Baseline: recommend the globally most popular products to everyone
pop = (train.groupBy("product_id").agg(F.sum("rating").alias("s")).orderBy(F.desc("s")).limit(K))
pop_list = [r.product_id for r in pop.collect()]
pop_df = relevant.filter(F.col("product_id").isin(pop_list)).groupBy("user_id").agg(F.count("*").alias("hits"))
base = n_rel.join(pop_df, "user_id", "left").fillna(0, subset=["hits"])
b = base.agg(F.avg(F.col("hits") / K).alias("p"), F.avg(F.col("hits") / F.col("n_rel")).alias("r")).collect()[0]
print(f"Popularity baseline -> Precision@{K}: {b['p']:.4f} | Recall@{K}: {b['r']:.4f}")

### 6.2 Cold-start fallback and a sample recommendation
New users have no history, so we fall back to the most popular products in the most viewed category (a simple hybrid approach).

In [ ]:
def recommend_for_user(uid, n=5):
    r = recs.filter(F.col("user_id") == uid).collect()
    if r:
        ids = [x.product_id for x in r[0].recommendations][:n]
        src = "ALS personalised"
    else:
        ids = pop_list[:n]
        src = "popularity fallback (cold start)"
    info = (df.select("product_id", "brand", "main_category", "price").dropDuplicates(["product_id"])
              .filter(F.col("product_id").isin(ids)))
    print(f"User {uid} -> {src}")
    info.show()

sample_user = recs.select("user_id").first()[0]
recommend_for_user(sample_user)
recommend_for_user(-1)   # unknown / new user

## 7. Scalability analysis
Runtime is measured with different **partition counts** and **data sizes**. On a real multi-node cluster the same code scales horizontally by adding executors.

In [ ]:
def bench(frame, label):
    t = time.time()
    frame.groupBy("brand", "main_category").agg(F.count("*"), F.sum("price")).collect()
    return round(time.time() - t, 2)

part_res = {}
for p in [1, 2, 4, 8, 16]:
    part_res[p] = bench(df.repartition(p), f"{p} partitions")
print("Partitions vs time:", part_res)

size_res = {}
for frac in [0.1, 0.25, 0.5, 1.0]:
    sub = df.sample(False, frac, seed=1)
    size_res[f"{int(frac*100)}%"] = bench(sub, f"{frac}")
print("Data size vs time:", size_res)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(list(part_res.keys()), list(part_res.values()), marker="o"); ax[0].set_title("Runtime vs partitions"); ax[0].set_xlabel("partitions"); ax[0].set_ylabel("seconds")
ax[1].plot(list(size_res.keys()), list(size_res.values()), marker="o", color="#d9534f"); ax[1].set_title("Runtime vs data size"); ax[1].set_xlabel("data fraction")
plt.tight_layout(); plt.show()

# Caching benefit
t1 = time.time(); df.unpersist(); df.groupBy("brand").count().collect(); no_cache = time.time() - t1
df.cache(); df.count()
t2 = time.time(); df.groupBy("brand").count().collect(); with_cache = time.time() - t2
print(f"Without cache: {no_cache:.2f}s | With cache: {with_cache:.2f}s")

## 8. Bonus: Structured Streaming (Velocity)
Simulates a live clickstream by reading Parquet files as a stream and counting events per type in rolling windows.

In [ ]:
stream_src = (spark.readStream.schema(spark.read.parquet(f"{LAKE}/clean_events").schema)
                .option("maxFilesPerTrigger", 2).parquet(f"{LAKE}/clean_events"))

live = (stream_src.withWatermark("event_time", "1 day")
          .groupBy(F.window("event_time", "1 hour"), "event_type").count())

q = (live.writeStream.outputMode("complete").format("memory").queryName("live_counts").start())
time.sleep(15)
spark.sql("SELECT window.start AS window_start, event_type, count FROM live_counts ORDER BY window_start DESC LIMIT 10").show(truncate=False)
q.stop()

## 9. Export results (for dashboard / serving layer)
Recommendations and segments are saved as Parquet. In production they would be served from MongoDB/Redis through an API.

In [ ]:
(recs.select("user_id", F.explode("recommendations").alias("rec"))
     .select("user_id", F.col("rec.product_id").alias("product_id"), F.col("rec.rating").alias("score"))
     .write.mode("overwrite").parquet(f"{LAKE}/recommendations"))

seg.select("user_id", "recency", "frequency", "monetary", F.col("prediction").alias("segment")) \
   .write.mode("overwrite").parquet(f"{LAKE}/segments")
print("Saved recommendations and segments to", LAKE)

In [ ]:
import os

print("Content folder:", os.listdir("/content"))
print("Sample data folder:", os.listdir("/content/sample_data"))

Content folder: ['.config', 'sample_data']
Sample data folder: ['anscombe.json', 'README.md', 'california_housing_test.csv', 'mnist_train_small.csv', 'california_housing_train.csv', 'mnist_test.csv']


## 10. Conclusion
- Extended the Experiment 1 case study into an end-to-end Big Data pipeline using PySpark.
- Parquet with partitioning reduced storage and improved query time compared to CSV.
- Spark SQL revealed the conversion funnel, cart abandonment, peak hours and top categories/brands.
- RFM + K-Means produced customer segments for targeted marketing.
- ALS produced personalised recommendations and beat the popularity baseline, with a cold-start fallback.
- Scalability tests showed how partitioning, data size and caching affect runtime; the same code runs on a multi-node cluster.

**Future scope:** real-time Kafka ingestion, deep-learning recommenders (two-tower, session-based), serving via MongoDB/Redis API, A/B testing of recommendations.